In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

df = pd.concat([
    pd.read_csv("../data/processed_22.csv"),
    pd.read_csv("../data/processed_23.csv"),
    pd.read_csv("../data/processed_24.csv"),
    pd.read_csv("../data/processed_25.csv")
], ignore_index=True)
df["date"] = pd.to_datetime(df["date"])

a = df.sort_values("Lpoly_expected_ml", ascending=False).head(10)[["Lpoly_expected_ml"]]
print(a)

     Lpoly_expected_ml
324         172.389686
323         131.889561
322          85.849369
476          75.519357
480          65.297662
487          61.108233
325          53.397145
481          48.266693
488          46.417918
321          44.784674


In [7]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def eval_reg(name, model, X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    r2_scores = []
    rmse_scores = []
    
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        r2_scores.append(r2_score(y_test, preds))
        rmse_scores.append(rmse(y_test, preds))
    
    return {
        "Model": name,
        "CV_R2_mean": np.mean(r2_scores),
        "CV_RMSE_mean": np.mean(rmse_scores),
    }

In [11]:
# Target: daily expected Lingulodinium polyedra per mL
y = df["Lpoly_expected_ml"].astype(float)
X = df.drop(columns=["date", "Lpoly_expected", "Lpoly_expected_ml"]).dropna(axis=1, how="all")

lin = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
ridge = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])

rf = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

perf = pd.DataFrame([
    eval_reg("Linear", lin, X, y),
    eval_reg("Ridge", ridge, X, y),
    eval_reg("RandomForest", rf, X, y),
])

print("Baseline timeseries CV performance")
print(perf)

Baseline timeseries CV performance
          Model     CV_R2_mean  CV_RMSE_mean
0        Linear -320424.835559   4280.381948
1         Ridge  -39683.543592    186.915361
2  RandomForest  -38609.887253     10.425445


In [14]:
# Feature importance

# random train-test split for interpretability (not time-based)
# not great but allows us to see which features are most good ish..
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 )

# 1) Ridge coefficients (standardized) = interpretable linear influence
ridge.fit(X_train, y_train)
coef = ridge.named_steps["model"].coef_
ridge_importance = pd.DataFrame({
    "feature": X.columns,
    "ridge_coef_abs": np.abs(coef),
    "ridge_coef_signed": coef
}).sort_values("ridge_coef_abs", ascending=False).reset_index(drop=True)

print("\nTop 10 Ridge coefficients (|coef|, standardized features)")
print(ridge_importance.head(10))

# 2) Random Forest impurity importance = nonlinear importance signal
rf.fit(X_train, y_train)
rf_importance = pd.DataFrame({
    "feature": X.columns,
    "rf_impurity_importance": rf.feature_importances_
}).sort_values("rf_impurity_importance", ascending=False).reset_index(drop=True)

print("\nTop 10 RandomForest impurity importances")
print(rf_importance.head(10))


Top 10 Ridge coefficients (|coef|, standardized features)
                                feature  ridge_coef_abs  ridge_coef_signed
0                                Ring05       10.779440         -10.779440
1                                 HOG34        8.320175           8.320175
2  summedConvexPerimeter_over_Perimeter        8.266895           8.266895
3                  texture_third_moment        8.086985          -8.086985
4            texture_average_gray_level        7.991270           7.991270
5                            ROI_per_ml        7.933867           7.933867
6                                 HOG17        7.269057           7.269057
7                                 HOG52        6.818484           6.818484
8             RotatedBoundingBox_xwidth        6.437335           6.437335
9          rotated_BoundingBox_solidity        6.349711          -6.349711

Top 10 RandomForest impurity importances
             feature  rf_impurity_importance
0  moment_invariant1         

In [ ]:
#Top 13 biological features only

# Explicitly define effort / metadata features to exclude
effort_features = ["ml_analyzed", "roiCount", "ROI_per_ml"]

top13_bio = [
    f for f in rf_importance["feature"].tolist()
    if f not in effort_features
][:13]

print("\nTop 13 biological features:")
print(top13_bio)

X_train_top13_bio = X_train[top13_bio]
X_test_top13_bio = X_test[top13_bio]

rf_top13_bio = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

rf_top13_bio.fit(X_train_top13_bio, y_train)

#Results
y_pred_top13_bio = rf_top13_bio.predict(X_test_top13_bio)

r2_top13_bio = r2_score(y_test, y_pred_top13_bio)
rmse_top13_bio = np.sqrt(np.mean((y_test - y_pred_top13_bio) ** 2))

print("\nReduced top-13 BIOLOGICAL RF performance")
print(f"Test R2:   {r2_top13_bio:.3f}")
print(f"Test RMSE: {rmse_top13_bio:.3f}")

rf_top13_bio_importance = pd.DataFrame({
    "feature": top13_bio,
    "rf_impurity_importance": rf_top13_bio.feature_importances_
}).sort_values("rf_impurity_importance", ascending=False)

print("\nReduced RF feature importance (biological only)")
print(rf_top13_bio_importance)


Top 13 biological features:
['moment_invariant1', 'Ring10', 'moment_invariant2', 'moment_invariant3', 'Ring09', 'HOG41', 'HOG79', 'moment_invariant7', 'RWhalfpowerintegral', 'HOG40', 'HOG18', 'HOG74', 'B90']

Reduced top-13 BIOLOGICAL RF performance
Test R2:   0.515
Test RMSE: 8.460

Reduced RF feature importance (biological only)
                feature  rf_impurity_importance
1                Ring10                0.222085
0     moment_invariant1                0.180003
4                Ring09                0.119572
7     moment_invariant7                0.090643
5                 HOG41                0.085153
2     moment_invariant2                0.075773
3     moment_invariant3                0.057788
6                 HOG79                0.045803
8   RWhalfpowerintegral                0.033520
10                HOG18                0.032996
12                  B90                0.021489
9                 HOG40                0.017819
11                HOG74                0.0